# Banking sector：A0--A9 × 5 RL agents forecast-to-policy best-policy ablation

本 notebook 用于生成与 healthcare innovation sector 表格一致的 bank 组 forecast-to-policy 消融实验。

核心输出：

```text
D:/paper/onging/ablation/rl_bank_best_policy/rl_ablation_raw_results.csv
D:/paper/onging/ablation/rl_bank_best_policy/rl_ablation_paper_table.csv
D:/paper/onging/ablation/rl_bank_best_policy/best_agent_by_variant.csv
D:/paper/onging/ablation/rl_bank_best_policy/bank_forecast_to_policy_best_policy_table.tex
```

表格口径：每个 A0--A9 预测状态变体，在 A2C/PPO/SAC/TD3/DDPG 以及 Buy-and-Hold、Equal Weight 中，按 FinalAccountValue 选择最优 downstream policy。


In [1]:
# ===============================
# 0. Basic imports and global paths
# ===============================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # notebook中也可保存图片；如需交互显示，可注释本行
import matplotlib.pyplot as plt

# ==============================================================
# FinRL 0.3.4 compatibility imports
# --------------------------------------------------------------
# 当前环境 pip list 显示：
#   finrl==0.3.4
#   finrl-meta==0.3.6
# 这个环境通常不能使用 finrl.meta.* 路径。
# 所以本 notebook 只使用旧版 FinRL 路径：
#   finrl.preprocessing.preprocessors
#   finrl.model.models
# ==============================================================

try:
    from finrl.preprocessing.preprocessors import FeatureEngineer
    print("FeatureEngineer import: finrl.preprocessing.preprocessors")
except Exception as e:
    raise ImportError(
        "当前环境无法导入 finrl.preprocessing.preprocessors.FeatureEngineer。\n"
        "请先在当前 Jupyter kernel 中运行：python -c \"import finrl, sys; print(finrl.__file__, sys.executable)\"\n"
        "确认 notebook 使用的是包含 finrl==0.3.4 的环境。"
    ) from e

try:
    from finrl.preprocessing.preprocessors import data_split as _finrl_data_split
    print("data_split import: finrl.preprocessing.preprocessors")
except Exception:
    _finrl_data_split = None
    print("data_split import: custom fallback data_split")

try:
    from finrl.model.models import DRLAgent
    print("DRLAgent import: finrl.model.models")
except Exception as e:
    raise ImportError(
        "当前环境无法导入 finrl.model.models.DRLAgent。\n"
        "这通常说明 Jupyter kernel 不是你的 FinRL 环境，或 FinRL 安装不完整。"
    ) from e


def data_split(df, start, end):
    """Compatible data_split wrapper.

    优先调用 FinRL 0.3.4 自带 data_split；如果当前版本没有，则使用本地实现。
    返回值会按 date/tic 排序，并把 index 重设为整数 day index，适配 StockPortfolioEnv。
    """
    if _finrl_data_split is not None:
        return _finrl_data_split(df, start, end)

    out = df[(df["date"] >= start) & (df["date"] < end)].copy()
    out = out.sort_values(["date", "tic"]).copy()
    out.index = pd.factorize(out["date"])[0]
    return out


# Gym / Stable-Baselines3 imports
# stable_baselines3==2.x 会优先使用 gymnasium；你的环境同时安装了 gym 和 Shimmy，
# 因此这里继续使用 gym.Env，一般会由 Shimmy 自动兼容。
import gym
from gym import spaces
from gym.utils import seeding
from stable_baselines3.common.vec_env import DummyVecEnv

# -------------------------------
# Project paths
# -------------------------------
PROJECT_ROOT = Path("D:/paper/onging")
GRAPH_DATA_DIR = PROJECT_ROOT / "data" / "graph"
ABLATION_ROOT = PROJECT_ROOT / "ablation"
RL_ABLATION_DIR = ABLATION_ROOT / "rl_bank_best_policy"
PRED_ABLATION_DIR = ABLATION_ROOT                 # 第一个预测消融 notebook 的输出目录
GENERATED_FINRL_DIR = ABLATION_ROOT / "finrl_csv_bank_best_policy" # 自动生成的 A0-A9 FinRL 长表 CSV 目录
BANK_CLOSE_CSV = PROJECT_ROOT / "data" / "bank" / "bankclose.csv"

RL_ABLATION_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_FINRL_DIR.mkdir(parents=True, exist_ok=True)

# 原始 Full MGT-FinRL 使用的数据
BASE_FULL_MGT_CSV = GRAPH_DATA_DIR / "banktrademergeGNNtr20241231.CSV"

# 与第一个预测消融 notebook 的模型编号保持一致
ALL_VARIANTS = [
    "A0_Full_MGT",
    "A1_Transformer_only",
    "A2_MGNN_only",
    "A3_SH_only",
    "A4_QA_only",
    "A5_KD_only",
    "A6_SH_QA",
    "A7_SH_KD",
    "A8_QA_KD",
    "A9_Fixed_fusion",
]

print("PRED_ABLATION_DIR =", PRED_ABLATION_DIR)
print("GENERATED_FINRL_DIR =", GENERATED_FINRL_DIR)
print("RL_ABLATION_DIR =", RL_ABLATION_DIR)
print("BANK_CLOSE_CSV =", BANK_CLOSE_CSV)
print("BASE_FULL_MGT_CSV =", BASE_FULL_MGT_CSV)


FeatureEngineer import: finrl.preprocessing.preprocessors
data_split import: custom fallback data_split
DRLAgent import: finrl.model.models
PRED_ABLATION_DIR = D:\paper\onging\ablation
GENERATED_FINRL_DIR = D:\paper\onging\ablation\finrl_csv_bank_best_policy
RL_ABLATION_DIR = D:\paper\onging\ablation\rl_bank_best_policy
BANK_CLOSE_CSV = D:\paper\onging\data\bank\bankclose.csv
BASE_FULL_MGT_CSV = D:\paper\onging\data\graph\banktrademergeGNNtr20241231.CSV


## 1. 与第一个预测消融 notebook 对应的输入文件配置

下面的 `RL_ABLATION_INPUTS` 与第一个 notebook 的 `ABLATION_CONFIGS` 一一对应。

如果你还没有生成某个版本的 FinRL 长表 CSV，程序会自动跳过该版本，不会中断运行。

建议文件命名方式：

```text
D:/paper/onging/data/graph/banktrademergeGNNtr20241231_A0_Full_MGT.CSV
D:/paper/onging/data/graph/banktrademergeGNNtr20241231_A1_Transformer_only.CSV
...
```

如果当前只有原始 `banktrademergeGNNtr20241231.CSV`，那么默认只会运行 `A0_Full_MGT`。

In [2]:
# ===============================
# 1. Ablation input mapping
# ===============================

# 先定义每个版本对应的“自动生成 FinRL 长表 CSV”路径。
# 后面 v5 自动转换 cell 会尝试从 D:/paper/onging/ablation/A*_seed*_pred.csv 生成这些文件。
def generated_finrl_csv_path(tag: str) -> Path:
    return GENERATED_FINRL_DIR / f"banktrademergeGNNtr20241231_{tag}.CSV"


def resolve_variant_csv(tag: str) -> Path:
    """优先使用自动生成的 A0-A9 长表 CSV；若 A0 未生成，则回退到原始 Full MGT 文件。"""
    gen_path = generated_finrl_csv_path(tag)
    if gen_path.exists():
        return gen_path
    if tag == "A0_Full_MGT" and BASE_FULL_MGT_CSV.exists():
        return BASE_FULL_MGT_CSV
    return gen_path


RL_ABLATION_INPUTS = {tag: resolve_variant_csv(tag) for tag in ALL_VARIANTS}

print("Initial ablation CSV mapping:")
for tag, path in RL_ABLATION_INPUTS.items():
    print(f"{tag:28s} -> {path} | exists={Path(path).exists()}")


Initial ablation CSV mapping:
A0_Full_MGT                  -> D:\paper\onging\data\graph\banktrademergeGNNtr20241231.CSV | exists=True
A1_Transformer_only          -> D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A1_Transformer_only.CSV | exists=False
A2_MGNN_only                 -> D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A2_MGNN_only.CSV | exists=False
A3_SH_only                   -> D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A3_SH_only.CSV | exists=False
A4_QA_only                   -> D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A4_QA_only.CSV | exists=False
A5_KD_only                   -> D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A5_KD_only.CSV | exists=False
A6_SH_QA                     -> D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A6_SH_QA.CSV | exists=False
A7_S

## 2. 实验设置

主文建议采用：

- 初始资金：`1,000,000`
- 交易成本：`0.0001`，即万分之一
- 交易区间：`2023-01-03` 到 `2024-12-31`
- 消融实验固定一个 RL agent，例如 `A2C`
- 完整策略比较再跑 PPO/A2C/SAC/TD3/DDPG

为了先跑通代码，默认只用 `SEEDS = [1]`。论文最终结果建议改成 `SEEDS = [1, 2, 3]`。

In [3]:
# ===============================
# 2. Experiment settings
# ===============================

# --------------------------------------------------------------
# 时间区间设置
# --------------------------------------------------------------
# 重要：第一个预测消融 notebook 当前输出的 pred.csv 一般是 2022 测试集预测。
# 因此这里默认用 2022 做策略层消融，保证预测矩阵行数和交易日期一致。
# 如果你已经重新生成 2023-2024 的预测结果，再把下面四个日期改为 2023-2024。
TRAIN_START = "2018-01-02"
TRAIN_END   = "2021-12-31"

PREDICTION_START = "2022-01-01"
PREDICTION_END   = "2022-12-31"

TRADE_START = "2022-01-01"
TRADE_END   = "2022-12-31"

# --------------------------------------------------------------
# 交易环境设置
# --------------------------------------------------------------
INITIAL_AMOUNT = 1_000_000
TRANSACTION_COST_PCT = 0.0001   # 万分之一交易成本
REWARD_SCALING = 1e-1
LOOKBACK = 252

TECH_INDICATOR_LIST = ["macd", "rsi_30", "cci_30", "dx_30"]

# --------------------------------------------------------------
# 消融实验设置
# --------------------------------------------------------------
# 按你的要求：A0-A9 每个预测版本都跑五个 RL agent。
AGENTS_FOR_ABLATION = ["a2c", "ppo", "sac", "td3", "ddpg"]

# 当前默认使用三个预测 seed 的平均值作为交易状态输入，减少 RL 训练次数。
# 如果想只用 seed1，则改成 1；如果想用 seed2，则改成 2。
PRED_SEED_MODE = "mean"

# RL 随机种子。先用 [1] 跑通；论文最终更稳健可改为 [1, 2, 3]，但计算量会变成 150 次训练。
SEEDS = [1]

# 自动从 A*_seed*_pred.csv 生成 FinRL 长表 CSV。
AUTO_GENERATE_VARIANT_FINRL_CSVS = True

# 主文需要跑的预测消融版本
VARIANTS_TO_RUN = [
    "A0_Full_MGT",
    "A1_Transformer_only",
    "A2_MGNN_only",
    "A3_SH_only",
    "A4_QA_only",
    "A5_KD_only",
    "A6_SH_QA",
    "A7_SH_KD",
    "A8_QA_KD",
    "A9_Fixed_fusion",
]

# --------------------------------------------------------------
# RL 训练步数
# --------------------------------------------------------------
# 若只是测试流程，可以临时把每个算法改成 5000；正式结果再恢复。
TOTAL_TIMESTEPS = {
    "a2c": 40000,
    "ppo": 40000,
    "sac": 60000,
    "td3": 50000,
    "ddpg": 50000,
}

MODEL_PARAMS = {
    "a2c": {
        "n_steps": 10,
        "ent_coef": 0.005,
        "learning_rate": 0.0004,
    },
    "ppo": {
        "n_steps": 2048,
        "ent_coef": 0.005,
        "learning_rate": 0.00025,
        "batch_size": 128,
    },
    "sac": {
        "batch_size": 128,
        "buffer_size": 1_000_000,
        "learning_rate": 0.0001,
        "learning_starts": 100,
        "ent_coef": "auto_0.1",
    },
    "td3": {
        "batch_size": 128,
        "buffer_size": 1_000_000,
        "learning_rate": 0.0001,
        "learning_starts": 100,
    },
    "ddpg": {
        "batch_size": 128,
        "buffer_size": 1_000_000,
        "learning_rate": 0.0001,
        "learning_starts": 100,
    },
}

print("TRAIN:", TRAIN_START, "~", TRAIN_END)
print("PREDICTION:", PREDICTION_START, "~", PREDICTION_END)
print("TRADE:", TRADE_START, "~", TRADE_END)
print("AGENTS_FOR_ABLATION:", AGENTS_FOR_ABLATION)
print("VARIANTS_TO_RUN:", VARIANTS_TO_RUN)
print("PRED_SEED_MODE:", PRED_SEED_MODE)
print("SEEDS:", SEEDS)


TRAIN: 2018-01-02 ~ 2021-12-31
PREDICTION: 2022-01-01 ~ 2022-12-31
TRADE: 2022-01-01 ~ 2022-12-31
AGENTS_FOR_ABLATION: ['a2c', 'ppo', 'sac', 'td3', 'ddpg']
VARIANTS_TO_RUN: ['A0_Full_MGT', 'A1_Transformer_only', 'A2_MGNN_only', 'A3_SH_only', 'A4_QA_only', 'A5_KD_only', 'A6_SH_QA', 'A7_SH_KD', 'A8_QA_KD', 'A9_Fixed_fusion']
PRED_SEED_MODE: mean
SEEDS: [1]


## 3. 数据读取与协方差状态构造

这部分替代原 notebook 中重复的 `FeatureEngineer` 和重复的 covariance 代码。

修正点：

1. 不再重复调用 `fe.preprocess_data(df)`；
2. 如果技术指标已经存在，则不重复生成；
3. 每次重新计算 `cov_list`，避免从 CSV 读取后变成字符串；
4. 训练集和交易集都会重新设置整数 day index，保证环境中的 `df.loc[self.day, :]` 正常工作。

In [4]:
# ===============================
# 3. Data preprocessing
# ===============================

def _reset_day_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["date", "tic"]).copy()
    df.index = pd.factorize(df["date"])[0]
    return df

def _check_required_columns(df: pd.DataFrame, csv_path: Path):
    required = {"date", "tic", "close"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} 缺少必要字段: {missing}")

def load_finrl_data(csv_path, lookback=252, tech_indicator_list=None):
    """
    读取 FinRL 长表 CSV，并生成技术指标和 rolling covariance state。

    输入 CSV 至少应包含:
    date, tic, open, high, low, close, volume

    返回:
    df with columns: date, tic, OHLCV, technical indicators, cov_list, return_list
    """
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"找不到数据文件: {csv_path}")

    df = pd.read_csv(csv_path)
    _check_required_columns(df, csv_path)

    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df = df.sort_values(["date", "tic"]).reset_index(drop=True)

    if tech_indicator_list is None:
        tech_indicator_list = TECH_INDICATOR_LIST

    # 如果技术指标缺失，则调用 FeatureEngineer
    missing_tech = [c for c in tech_indicator_list if c not in df.columns]
    if missing_tech:
        print(f"[{csv_path.name}] Missing technical indicators {missing_tech}; running FeatureEngineer...")
        fe = FeatureEngineer(
            use_technical_indicator=True,
            use_turbulence=False,
            user_defined_feature=False
        )
        df = fe.preprocess_data(df)
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    else:
        print(f"[{csv_path.name}] Technical indicators already exist; skip FeatureEngineer.")

    # 避免从 CSV 读取旧 cov_list/return_list 后变成字符串
    for col in ["cov_list", "return_list"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = df.sort_values(["date", "tic"], ignore_index=True)
    df.index = pd.factorize(df["date"])[0]

    unique_dates = df["date"].unique()
    if len(unique_dates) <= lookback:
        raise ValueError(f"数据交易日数量 {len(unique_dates)} 小于等于 lookback={lookback}，无法构造协方差状态。")

    cov_list = []
    return_list = []

    for i in range(lookback, len(unique_dates)):
        data_lookback = df.loc[i - lookback:i, :]
        price_lookback = data_lookback.pivot_table(index="date", columns="tic", values="close")
        return_lookback = price_lookback.pct_change().dropna()

        return_list.append(return_lookback)
        cov_list.append(return_lookback.cov().values)

    df_cov = pd.DataFrame({
        "date": unique_dates[lookback:],
        "cov_list": cov_list,
        "return_list": return_list
    })

    df = df.reset_index(drop=True)
    df = df.merge(df_cov, on="date", how="inner")
    df = df.sort_values(["date", "tic"]).reset_index(drop=True)

    return df

def make_train_trade(df: pd.DataFrame):
    train = data_split(df, TRAIN_START, TRAIN_END)
    trade = data_split(df, TRADE_START, TRADE_END)

    train = _reset_day_index(train)
    trade = _reset_day_index(trade)

    return train, trade


## 3.1 v5：从第一个预测消融结果自动生成 FinRL 长表 CSV

第一个 notebook 在 `D:/paper/onging/ablation` 中保存的是：

```text
A0_Full_MGT_seed1_pred.csv
A0_Full_MGT_seed2_pred.csv
...
A9_Fixed_fusion_seed3_pred.csv
```

这些文件只是预测矩阵，不能直接给 FinRL 使用。本 cell 会：

1. 读取预测矩阵；
2. 用 `bankclose.csv` 的 MinMaxScaler 反归一化回真实价格；
3. 将预测 close 合并回 `banktrademergeGNNtr20241231.CSV`；
4. 生成 `D:/paper/onging/ablation/finrl_csv/banktrademergeGNNtr20241231_A*.CSV`；
5. 自动刷新 `RL_ABLATION_INPUTS`，后面的五个 RL 算法会读取这些文件。

如果这里提示“预测矩阵行数与日期数量不一致”，说明第一个预测 notebook 输出的日期区间和这里的 `PREDICTION_START/PREDICTION_END` 不一致。

In [5]:
# ===============================
# 3.1 Auto-generate FinRL CSVs from prediction ablation outputs
# ===============================

from sklearn.preprocessing import MinMaxScaler


def load_close_matrix_and_scaler():
    """读取第一个预测 notebook 使用的 bankclose.csv，并复现 MinMaxScaler(feature_range=(-1,1))."""
    if not BANK_CLOSE_CSV.exists():
        raise FileNotFoundError(f"找不到 BANK_CLOSE_CSV: {BANK_CLOSE_CSV}")

    close = pd.read_csv(BANK_CLOSE_CSV, index_col=0)
    close.index = pd.to_datetime(close.index).strftime("%Y-%m-%d")
    close = close.sort_index()
    close = close.apply(pd.to_numeric, errors="coerce")

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(close.values)
    return close, scaler


def find_prediction_files(tag: str):
    """
    兼容两类预测文件命名：
    1) *_pred_raw.csv：已经反归一化到真实价格尺度，优先使用；
    2) *_pred.csv：通常是 MinMaxScaler(feature_range=(-1,1)) 后的预测值，需要 inverse_transform。
    """
    raw_files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_pred_raw.csv"))
    if raw_files:
        return raw_files
    files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_pred.csv"))
    return files


def read_prediction_matrix(tag: str, seed_mode="mean"):
    """读取某个消融版本的预测矩阵。seed_mode='mean' 对所有 seed 取平均；seed_mode=1 读取 seed1。"""
    if seed_mode == "mean":
        files = find_prediction_files(tag)
        if not files:
            raise FileNotFoundError(f"找不到 {tag}_seed*_pred.csv，请检查 {PRED_ABLATION_DIR}")
        mats = []
        for f in files:
            mat = pd.read_csv(f)
            unnamed = [c for c in mat.columns if str(c).lower().startswith("unnamed")]
            if unnamed:
                mat = mat.drop(columns=unnamed)
            mats.append(mat.astype(float).values)
        shapes = {m.shape for m in mats}
        if len(shapes) != 1:
            raise ValueError(f"{tag} 的多个 seed 预测矩阵 shape 不一致: {shapes}")
        return np.mean(np.stack(mats, axis=0), axis=0), files

    # seed_mode 是整数或字符串数字
    seed = int(seed_mode)
    f = PRED_ABLATION_DIR / f"{tag}_seed{seed}_pred.csv"
    if not f.exists():
        raise FileNotFoundError(f"找不到预测文件: {f}")
    mat = pd.read_csv(f)
    unnamed = [c for c in mat.columns if str(c).lower().startswith("unnamed")]
    if unnamed:
        mat = mat.drop(columns=unnamed)
    return mat.astype(float).values, [f]


def prediction_dates_from_close(close_df: pd.DataFrame, n_rows: int, start_date: str, end_date: str):
    dates = pd.Series(close_df.index, name="date")
    mask = (dates >= start_date) & (dates <= end_date)
    selected = dates.loc[mask].tolist()

    if len(selected) != n_rows:
        raise ValueError(
            "预测矩阵行数与指定日期区间不一致。\n"
            f"  pred rows        = {n_rows}\n"
            f"  selected dates    = {len(selected)}\n"
            f"  date range        = {start_date} ~ {end_date}\n"
            "解决方法：\n"
            "  1) 如果第一个预测 notebook 当前只输出 2022 测试集，则把 PREDICTION_START/PREDICTION_END 改成 2022-01-01/2022-12-31；\n"
            "  2) 如果要做 2023-2024 投资回测，则需要先在第一个预测 notebook 中生成 2023-2024 的 trade_pred.csv。"
        )
    return selected


def build_variant_finrl_csv_from_pred(tag: str, seed_mode="mean", start_date=None, end_date=None):
    """把 A*_seed*_pred.csv 转成 FinRL 长表 CSV。"""
    start_date = start_date or PREDICTION_START
    end_date = end_date or PREDICTION_END

    pred_scaled, used_files = read_prediction_matrix(tag, seed_mode=seed_mode)
    close_df, scaler = load_close_matrix_and_scaler()

    if pred_scaled.shape[1] != close_df.shape[1]:
        raise ValueError(
            f"{tag}: 预测列数 {pred_scaled.shape[1]} 与 bankclose 股票数 {close_df.shape[1]} 不一致。"
        )

    # 如果读取的是 *_pred_raw.csv，则认为已经是真实价格尺度，不再 inverse_transform；
    # 如果读取的是 *_pred.csv，则按 bankclose 的 MinMaxScaler 反归一化。
    used_raw_price = all("_pred_raw" in Path(p).name for p in used_files)
    if used_raw_price:
        pred_real = pred_scaled
    else:
        pred_real = scaler.inverse_transform(pred_scaled)
    tickers = close_df.columns.astype(str).tolist()
    dates = prediction_dates_from_close(close_df, pred_real.shape[0], start_date, end_date)

    pred_wide = pd.DataFrame(pred_real, index=dates, columns=tickers)
    pred_wide.index.name = "date"
    pred_long = pred_wide.reset_index().melt(id_vars="date", var_name="tic", value_name="pred_close")

    base = pd.read_csv(BASE_FULL_MGT_CSV)
    _check_required_columns(base, BASE_FULL_MGT_CSV)
    base["date"] = pd.to_datetime(base["date"]).dt.strftime("%Y-%m-%d")
    base["tic"] = base["tic"].astype(str)

    # ticker 对齐检查
    base_tickers = set(base["tic"].astype(str).unique())
    pred_tickers = set(tickers)
    if not pred_tickers.issubset(base_tickers):
        raise ValueError(
            f"{tag}: bankclose ticker 与 FinRL 长表 ticker 不一致。\n"
            f"bankclose not in base: {sorted(pred_tickers - base_tickers)[:10]}"
        )

    merged = base.merge(pred_long, on=["date", "tic"], how="left")
    replace_mask = merged["pred_close"].notna()
    merged.loc[replace_mask, "close"] = merged.loc[replace_mask, "pred_close"]
    merged = merged.drop(columns=["pred_close"])

    out_csv = generated_finrl_csv_path(tag)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_csv, index=False)

    print(f"[OK] {tag}: generated {out_csv}")
    print(f"     used files: {[p.name for p in used_files]}")
    print(f"     replaced close rows: {int(replace_mask.sum())}")
    return out_csv


def auto_generate_all_variant_finrl_csvs():
    generated = {}
    for tag in ALL_VARIANTS:
        try:
            generated[tag] = build_variant_finrl_csv_from_pred(
                tag=tag,
                seed_mode=PRED_SEED_MODE,
                start_date=PREDICTION_START,
                end_date=PREDICTION_END,
            )
        except Exception as e:
            print(f"[WARN] {tag}: cannot generate FinRL CSV -> {e}")
    return generated


if AUTO_GENERATE_VARIANT_FINRL_CSVS:
    print("\nStart auto-generating FinRL CSVs from prediction ablation outputs...")
    generated_paths = auto_generate_all_variant_finrl_csvs()
else:
    generated_paths = {}
    print("AUTO_GENERATE_VARIANT_FINRL_CSVS=False; skip auto generation.")

# 刷新映射：优先使用刚生成的 finrl_csv；A0 如果没生成，则回退原始 BASE_FULL_MGT_CSV
RL_ABLATION_INPUTS = {tag: resolve_variant_csv(tag) for tag in ALL_VARIANTS}

print("\nRefreshed ablation CSV mapping:")
for tag, path in RL_ABLATION_INPUTS.items():
    print(f"{tag:28s} -> {path} | exists={Path(path).exists()}")



Start auto-generating FinRL CSVs from prediction ablation outputs...
[OK] A0_Full_MGT: generated D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A0_Full_MGT.CSV
     used files: ['A0_Full_MGT_seed1_pred.csv', 'A0_Full_MGT_seed2_pred.csv', 'A0_Full_MGT_seed3_pred.csv']
     replaced close rows: 3630
[OK] A1_Transformer_only: generated D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A1_Transformer_only.CSV
     used files: ['A1_Transformer_only_seed1_pred.csv', 'A1_Transformer_only_seed2_pred.csv', 'A1_Transformer_only_seed3_pred.csv']
     replaced close rows: 3630
[OK] A2_MGNN_only: generated D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNNtr20241231_A2_MGNN_only.CSV
     used files: ['A2_MGNN_only_seed1_pred.csv', 'A2_MGNN_only_seed2_pred.csv', 'A2_MGNN_only_seed3_pred.csv']
     replaced close rows: 3630
[OK] A3_SH_only: generated D:\paper\onging\ablation\finrl_csv_bank_best_policy\banktrademergeGNN

## 4. 修正版 Portfolio Allocation Environment

相对于原 notebook，主要修正：

1. `transaction_cost_pct` 不再是 0，默认使用万分之一；
2. 在 `step()` 中显式计算 turnover 和 transaction cost；
3. 保存 `turnover_memory` 和 `cost_memory`；
4. reward 使用 `portfolio_return * reward_scaling`，避免直接用账户净值导致奖励尺度过大；
5. `save_asset_memory()` 输出 daily return 和 account value。

In [6]:
# ===============================
# 4. Portfolio allocation environment with transaction cost
# ===============================

class StockPortfolioEnv(gym.Env):
    metadata = {"render.modes": ["human"]}

    def __init__(
        self,
        df,
        stock_dim,
        hmax,
        initial_amount,
        transaction_cost_pct,
        reward_scaling,
        state_space,
        action_space,
        tech_indicator_list,
        turbulence_threshold=None,
        lookback=252,
        day=0,
    ):
        self.day = day
        self.lookback = lookback
        self.df = df
        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.transaction_cost_pct = transaction_cost_pct
        self.reward_scaling = reward_scaling
        self.state_space = state_space
        self.action_space_dim = action_space
        self.tech_indicator_list = tech_indicator_list
        self.turbulence_threshold = turbulence_threshold

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.action_space_dim,), dtype=np.float32)
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.state_space + len(self.tech_indicator_list), self.state_space),
            dtype=np.float32,
        )

        self.data = self.df.loc[self.day, :]
        self.covs = self.data["cov_list"].values[0]
        self.state = np.append(
            np.array(self.covs),
            [self.data[tech].values.tolist() for tech in self.tech_indicator_list],
            axis=0,
        )

        self.terminal = False
        self.portfolio_value = self.initial_amount
        self.reward = 0.0

        self.asset_memory = [self.initial_amount]
        self.portfolio_return_memory = [0.0]
        self.actions_memory = [[1 / self.stock_dim] * self.stock_dim]
        self.turnover_memory = [0.0]
        self.cost_memory = [0.0]
        self.date_memory = [self.data.date.unique()[0]]

    def step(self, actions):
        self.terminal = self.day >= len(self.df.index.unique()) - 1

        if self.terminal:
            return self.state.astype(np.float32), self.reward, self.terminal, {}

        # 当前动作 -> long-only simplex 权重
        weights = self.softmax_normalization(actions)
        prev_weights = np.array(self.actions_memory[-1], dtype=float)

        # 当前日价格记忆
        last_day_memory = self.data

        # 进入下一交易日
        self.day += 1
        self.data = self.df.loc[self.day, :]

        self.covs = self.data["cov_list"].values[0]
        self.state = np.append(
            np.array(self.covs),
            [self.data[tech].values.tolist() for tech in self.tech_indicator_list],
            axis=0,
        )

        # 组合收益、换手率和交易成本
        price_relative = self.data.close.values / last_day_memory.close.values
        gross_return = float(np.sum((price_relative - 1.0) * weights))

        turnover = float(np.sum(np.abs(weights - prev_weights)))
        cost = float(self.transaction_cost_pct * turnover)

        portfolio_return = gross_return - cost

        # 更新账户
        new_portfolio_value = self.portfolio_value * (1.0 + portfolio_return)
        self.portfolio_value = new_portfolio_value

        # 保存轨迹
        self.actions_memory.append(weights)
        self.turnover_memory.append(turnover)
        self.cost_memory.append(cost)
        self.portfolio_return_memory.append(portfolio_return)
        self.asset_memory.append(new_portfolio_value)
        self.date_memory.append(self.data.date.unique()[0])

        # 使用收益率作为 reward，避免账户值尺度过大
        self.reward = portfolio_return * self.reward_scaling

        return self.state.astype(np.float32), self.reward, self.terminal, {}

    def reset(self):
        self.asset_memory = [self.initial_amount]
        self.day = 0
        self.data = self.df.loc[self.day, :]

        self.covs = self.data["cov_list"].values[0]
        self.state = np.append(
            np.array(self.covs),
            [self.data[tech].values.tolist() for tech in self.tech_indicator_list],
            axis=0,
        )

        self.portfolio_value = self.initial_amount
        self.reward = 0.0
        self.terminal = False
        self.portfolio_return_memory = [0.0]
        self.actions_memory = [[1 / self.stock_dim] * self.stock_dim]
        self.turnover_memory = [0.0]
        self.cost_memory = [0.0]
        self.date_memory = [self.data.date.unique()[0]]

        return self.state.astype(np.float32)

    def render(self, mode="human"):
        return self.state

    def softmax_normalization(self, actions):
        # SB3/DummyVecEnv may pass actions with shape (1, N); flatten to (N,)
        actions = np.array(actions, dtype=float).reshape(-1)
        actions = actions - np.max(actions)
        numerator = np.exp(actions)
        denominator = np.sum(numerator)
        if denominator == 0 or not np.isfinite(denominator):
            return np.ones(self.stock_dim) / self.stock_dim
        return numerator / denominator

    def save_asset_memory(self):
        return pd.DataFrame({
            "date": self.date_memory,
            "daily_return": self.portfolio_return_memory,
            "account_value": self.asset_memory,
            "turnover": self.turnover_memory,
            "transaction_cost": self.cost_memory,
        })

    def save_action_memory(self):
        df_date = pd.DataFrame(self.date_memory, columns=["date"])
        df_actions = pd.DataFrame(self.actions_memory)
        df_actions.columns = self.data.tic.values
        df_actions.index = df_date.date
        return df_actions

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def get_sb_env(self):
        e = DummyVecEnv([lambda: self])
        obs = e.reset()
        return e, obs


## 5. 评价指标

输出指标包括：

- CR：累计收益率
- AR：年化收益率
- Vol：年化波动率
- SR：Sharpe ratio
- MDD：最大回撤
- Calmar：Calmar ratio
- Sortino：Sortino ratio
- Omega：Omega ratio
- Turnover：平均换手率
- FinalAccountValue：期末账户价值

In [7]:
# ===============================
# 5. Metrics
# ===============================

def calculate_portfolio_metrics(df_daily_return: pd.DataFrame, df_actions: pd.DataFrame = None):
    r = df_daily_return["daily_return"].astype(float).values

    if len(r) == 0:
        raise ValueError("daily_return is empty.")

    cumulative_return = float(np.prod(1.0 + r) - 1.0)
    annual_return = float((1.0 + cumulative_return) ** (252 / max(len(r), 1)) - 1.0)
    annual_vol = float(np.std(r, ddof=1) * np.sqrt(252)) if len(r) > 1 else np.nan

    sharpe = annual_return / annual_vol if annual_vol and annual_vol != 0 else np.nan

    cumulative_curve = np.cumprod(1.0 + r)
    running_max = np.maximum.accumulate(cumulative_curve)
    drawdown = cumulative_curve / running_max - 1.0
    mdd = float(np.min(drawdown))

    calmar = annual_return / abs(mdd) if mdd != 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(np.std(downside, ddof=1) * np.sqrt(252)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol != 0 else np.nan

    pos_sum = float(np.sum(r[r > 0]))
    neg_sum = float(abs(np.sum(r[r < 0])))
    omega = pos_sum / neg_sum if neg_sum != 0 else np.inf

    if "account_value" in df_daily_return.columns:
        final_account_value = float(df_daily_return["account_value"].iloc[-1])
    else:
        final_account_value = float(cumulative_curve[-1])

    if "turnover" in df_daily_return.columns:
        turnover = float(df_daily_return["turnover"].mean())
    elif df_actions is not None:
        turnover = float(df_actions.diff().abs().sum(axis=1).mean())
    else:
        turnover = np.nan

    return {
        "CR": cumulative_return,
        "AR": annual_return,
        "Vol": annual_vol,
        "SR": sharpe,
        "MDD": mdd,
        "Calmar": calmar,
        "Sortino": sortino,
        "Omega": omega,
        "Turnover": turnover,
        "FinalAccountValue": final_account_value,
    }


In [8]:

# ===============================
# 5.1 Normalize FinRL prediction outputs
# ===============================

def normalize_daily_return_df(df_daily_return):
    """Make returned daily_return dataframe robust across FinRL versions."""
    if isinstance(df_daily_return, list) and len(df_daily_return) > 0:
        df_daily_return = df_daily_return[0]
    df_daily_return = pd.DataFrame(df_daily_return).copy()

    # Some FinRL versions return account_value only
    if "daily_return" not in df_daily_return.columns:
        if "account_value" in df_daily_return.columns:
            df_daily_return["daily_return"] = (
                df_daily_return["account_value"].astype(float).pct_change().fillna(0.0)
            )
        else:
            raise ValueError(
                "df_daily_return must contain daily_return or account_value. "
                f"Columns={df_daily_return.columns.tolist()}"
            )

    if "account_value" not in df_daily_return.columns:
        df_daily_return["account_value"] = (
            INITIAL_AMOUNT * (1.0 + df_daily_return["daily_return"].astype(float)).cumprod()
        )

    if "date" not in df_daily_return.columns:
        df_daily_return.insert(0, "date", np.arange(len(df_daily_return)))

    return df_daily_return


def normalize_action_df(df_actions):
    """Make returned action dataframe robust across FinRL versions."""
    if isinstance(df_actions, list) and len(df_actions) > 0:
        df_actions = df_actions[0]
    df_actions = pd.DataFrame(df_actions).copy()
    return df_actions


## 6. 传统投资基准

这部分用于回应审稿人“缺少 objective investment baselines”的问题。

默认生成：

1. `EqualWeight_Rebalanced`：每日等权再平衡；
2. `BuyHold_EqualInitial`：初始等权买入并持有。

In [9]:
# ===============================
# 6. Objective investment baselines
# ===============================

def equal_weight_rebalanced_baseline(trade_df: pd.DataFrame):
    price = trade_df.pivot_table(index="date", columns="tic", values="close").sort_index()
    asset_ret = price.pct_change().fillna(0.0)
    daily_ret = asset_ret.mean(axis=1)

    df_ret = pd.DataFrame({
        "date": daily_ret.index,
        "daily_return": daily_ret.values,
    })
    df_ret["account_value"] = INITIAL_AMOUNT * (1.0 + df_ret["daily_return"]).cumprod()
    return df_ret

def buy_and_hold_equal_initial_baseline(trade_df: pd.DataFrame):
    price = trade_df.pivot_table(index="date", columns="tic", values="close").sort_index()
    rel_price = price / price.iloc[0]
    portfolio_curve = rel_price.mean(axis=1)
    daily_ret = portfolio_curve.pct_change().fillna(0.0)

    df_ret = pd.DataFrame({
        "date": daily_ret.index,
        "daily_return": daily_ret.values,
    })
    df_ret["account_value"] = INITIAL_AMOUNT * (1.0 + df_ret["daily_return"]).cumprod()
    return df_ret

def run_objective_baselines(trade_df: pd.DataFrame, tag: str):
    rows = []

    baselines = {
        "EqualWeight_Rebalanced": equal_weight_rebalanced_baseline(trade_df),
        "BuyHold_EqualInitial": buy_and_hold_equal_initial_baseline(trade_df),
    }

    for name, df_ret in baselines.items():
        out_path = RL_ABLATION_DIR / f"{tag}_{name}_daily_return.csv"
        df_ret.to_csv(out_path, index=False)

        metrics = calculate_portfolio_metrics(df_ret)
        metrics.update({
            "variant": tag,
            "agent": name,
            "seed": -1,
            "csv_path": "",
        })
        rows.append(metrics)

    return rows


## 7. 单个 RL 消融实验函数

这部分是主函数。

输入一个 FinRL 长表 CSV，训练指定 RL agent，并输出：

- daily return CSV
- action/weight CSV
- 单次实验指标字典

In [10]:
# ===============================
# 7. Run one RL experiment
# ===============================

def build_env_kwargs(train_df: pd.DataFrame):
    stock_dimension = len(train_df.tic.unique())
    state_space = stock_dimension

    env_kwargs = {
        "hmax": 100,
        "initial_amount": INITIAL_AMOUNT,
        "transaction_cost_pct": TRANSACTION_COST_PCT,
        "state_space": state_space,
        "stock_dim": stock_dimension,
        "tech_indicator_list": TECH_INDICATOR_LIST,
        "action_space": stock_dimension,
        "reward_scaling": REWARD_SCALING,
    }
    return env_kwargs

def train_and_trade_one_agent(csv_path, variant_tag, agent_name="a2c", seed=1):
    set_global_seed(seed)
    csv_path = Path(csv_path)

    print(f"\n========== {variant_tag} | {agent_name.upper()} | seed={seed} ==========")
    print("CSV:", csv_path)

    df = load_finrl_data(csv_path, lookback=LOOKBACK, tech_indicator_list=TECH_INDICATOR_LIST)
    train, trade = make_train_trade(df)

    if train.empty:
        raise ValueError(f"{variant_tag}: train 数据为空，请检查日期范围 {TRAIN_START} 到 {TRAIN_END}")
    if trade.empty:
        raise ValueError(f"{variant_tag}: trade 数据为空，请检查日期范围 {TRADE_START} 到 {TRADE_END}")

    env_kwargs = build_env_kwargs(train)

    e_train_gym = StockPortfolioEnv(df=train, **env_kwargs)
    env_train, _ = e_train_gym.get_sb_env()

    agent = DRLAgent(env=env_train)

    params = MODEL_PARAMS.get(agent_name, {}).copy()
    # Stable-Baselines3 大多数算法支持 seed 参数
    params["seed"] = seed

    model = agent.get_model(model_name=agent_name, model_kwargs=params)
    trained_model = agent.train_model(
        model=model,
        tb_log_name=f"{variant_tag}_{agent_name}_seed{seed}",
        total_timesteps=TOTAL_TIMESTEPS[agent_name],
    )

    e_trade_gym = StockPortfolioEnv(df=trade, **env_kwargs)

    df_daily_return, df_actions = DRLAgent.DRL_prediction(
        model=trained_model,
        environment=e_trade_gym,
    )

    # 重要修正：
    # 不要在 DRL_prediction 之后再调用 e_trade_gym.save_asset_memory() 覆盖结果。
    # DummyVecEnv 在 done=True 后可能自动 reset 环境，导致 memory 只剩初始值，
    # 进而出现 A2C 的 CR=0、AR=0、Turnover=0 的假结果。
    df_daily_return = normalize_daily_return_df(df_daily_return)
    df_actions = normalize_action_df(df_actions)

    metrics = calculate_portfolio_metrics(df_daily_return, df_actions)
    metrics.update({
        "variant": variant_tag,
        "agent": agent_name,
        "seed": seed,
        "csv_path": str(csv_path),
    })

    # Save outputs
    daily_path = RL_ABLATION_DIR / f"{variant_tag}_{agent_name}_seed{seed}_daily_return.csv"
    action_path = RL_ABLATION_DIR / f"{variant_tag}_{agent_name}_seed{seed}_actions.csv"

    df_daily_return.to_csv(daily_path, index=False)
    df_actions.to_csv(action_path)

    print("Saved:", daily_path)
    print("Saved:", action_path)
    print(metrics)

    return metrics


## 8. 批量运行 forecast-to-policy 消融实验

默认只运行 `VARIANTS_TO_RUN` 中存在 CSV 的版本。

如果某个 CSV 不存在，会显示 `[SKIP]`，不会报错。

In [12]:
# ===============================
# 8. Batch run RL ablation
# ===============================

all_metrics = []

for variant_tag in VARIANTS_TO_RUN:
    csv_path = RL_ABLATION_INPUTS[variant_tag]

    if not Path(csv_path).exists():
        print(f"[SKIP] {variant_tag}: file not found -> {csv_path}")
        continue

    # 对每个存在的版本，先加两个客观投资基准
    try:
        df_for_baseline = load_finrl_data(csv_path, lookback=LOOKBACK, tech_indicator_list=TECH_INDICATOR_LIST)
        _, trade_for_baseline = make_train_trade(df_for_baseline)
        baseline_rows = run_objective_baselines(trade_for_baseline, variant_tag)
        all_metrics.extend(baseline_rows)
    except Exception as e:
        print(f"[Warning] Objective baselines failed for {variant_tag}: {e}")

    # RL forecast-to-policy ablation
    for agent_name in AGENTS_FOR_ABLATION:
        for seed in SEEDS:
            try:
                metrics = train_and_trade_one_agent(
                    csv_path=csv_path,
                    variant_tag=variant_tag,
                    agent_name=agent_name,
                    seed=seed,
                )
                all_metrics.append(metrics)
            except Exception as e:
                print(f"[ERROR] {variant_tag} | {agent_name} | seed={seed}: {e}")

raw_metrics_df = pd.DataFrame(all_metrics)
raw_metrics_path = RL_ABLATION_DIR / "rl_ablation_raw_results.csv"
raw_metrics_df.to_csv(raw_metrics_path, index=False)

raw_metrics_df


[banktrademergeGNNtr20241231_A0_Full_MGT.CSV] Missing technical indicators ['macd', 'rsi_30', 'cci_30', 'dx_30']; running FeatureEngineer...


C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

Successfully added technical indicators
[ERROR] A0_Full_MGT | a2c | seed=1: name 'set_global_seed' is not defined
[ERROR] A0_Full_MGT | ppo | seed=1: name 'set_global_seed' is not defined
[ERROR] A0_Full_MGT | sac | seed=1: name 'set_global_seed' is not defined
[ERROR] A0_Full_MGT | td3 | seed=1: name 'set_global_seed' is not defined
[ERROR] A0_Full_MGT | ddpg | seed=1: name 'set_global_seed' is not defined
[banktrademergeGNNtr20241231_A1_Transformer_only.CSV] Missing technical indicators ['macd', 'rsi_30', 'cci_30', 'dx_30']; running FeatureEngineer...


...

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(


Successfully added technical indicators
[ERROR] A9_Fixed_fusion | a2c | seed=1: name 'set_global_seed' is not defined
[ERROR] A9_Fixed_fusion | ppo | seed=1: name 'set_global_seed' is not defined
[ERROR] A9_Fixed_fusion | sac | seed=1: name 'set_global_seed' is not defined
[ERROR] A9_Fixed_fusion | td3 | seed=1: name 'set_global_seed' is not defined
[ERROR] A9_Fixed_fusion | ddpg | seed=1: name 'set_global_seed' is not defined


,CR,AR,Vol,SR,MDD,Calmar,Sortino,Omega,Turnover,FinalAccountValue,variant,agent,seed,csv_path
0,0.067378,0.070258,0.134722,0.521504,-0.081513,0.861924,0.834250,1.114313,NaN,1.067378e+06,A0_Full_MGT,EqualWeight_Rebalanced,-1,
1,0.032440,0.033803,0.134256,0.251783,-0.094951,0.356008,0.391118,1.061263,NaN,1.032440e+06,A0_Full_MGT,BuyHold_EqualInitial,-1,
2,-0.018385,-0.019137,0.032308,-0.592338,-0.051036,-0.374974,-0.936791,0.907433,NaN,9.816151e+05,A1_Transformer_only,EqualWeight_Rebalanced,-1,
3,-0.018980,-0.019757,0.031825,-0.620783,-0.050688,-0.389773,-0.977704,0.903164,NaN,9.810199e+05,A1_Transformer_only,BuyHold_EqualInitial,-1,
4,0.078524,0.081898,0.201579,0.406283,-0.081706,1.002354,0.572866,1.092955,NaN,1.078524e+06,A2_MGNN_only,EqualWeight_Rebalanced,-1,
5,-0.012545,-0.013060,0.204793,-0.063770,-0.111082,-0.117567,-0.088290,1.006827,NaN,9.874554e+05,A2_MGNN_only,BuyHold_EqualInitial,-1,
6,-0.055871,-0.058111,0.089112,-0.652117,-0.140674,-0.413093,-1.022832,0.901343,NaN,9.441289e+05,A3_SH_only,EqualWeight_Rebalanced,-1,
7,-0.058877,-0.061234,0.087679,-0.698386,-0.140163,-0.436877,-1.088600,0.894062,NaN,9.411229e+05,A3_SH_only,BuyHold_EqualInitial,-1,
8,-0.055718,-0.057953,0.115399,-0.502193,-0.140914,-0.411262,-0.797862,0.924819,NaN,9.442816e+05,A4_QA_only,EqualWeight_Rebalanced,-1,
9,-0.076315,-0.079340,0.111273,-0.713022,-0.150357,-0.527679,-1.118736,0.889598,NaN,9.236850e+05,A4_QA_only,BuyHold_EqualInitial,-1,


## 9. 汇总结果表

生成论文中可用的 `mean ± std` 汇总表。

In [13]:
# ===============================
# 9. Summary table
# ===============================

if len(raw_metrics_df) == 0:
    print("No results. Please check RL_ABLATION_INPUTS and CSV paths.")
else:
    metric_cols = ["CR", "AR", "Vol", "SR", "MDD", "Calmar", "Sortino", "Omega", "Turnover", "FinalAccountValue"]

    summary_df = raw_metrics_df.groupby(["variant", "agent"])[metric_cols].agg(["mean", "std"])

    summary_path = RL_ABLATION_DIR / "rl_ablation_summary.csv"
    summary_df.to_csv(summary_path)

    print("Saved summary:", summary_path)
    display(summary_df)


Saved summary: D:\paper\onging\ablation\rl_bank_best_policy\rl_ablation_summary.csv


CR            AR      \
                                                mean std      mean std   
variant             agent                                                
A0_Full_MGT         BuyHold_EqualInitial    0.032440 NaN  0.033803 NaN   
                    EqualWeight_Rebalanced  0.067378 NaN  0.070258 NaN   
A1_Transformer_only BuyHold_EqualInitial   -0.018980 NaN -0.019757 NaN   
                    EqualWeight_Rebalanced -0.018385 NaN -0.019137 NaN   
A2_MGNN_only        BuyHold_EqualInitial   -0.012545 NaN -0.013060 NaN   
                    EqualWeight_Rebalanced  0.078524 NaN  0.081898 NaN   
A3_SH_only          BuyHold_EqualInitial   -0.058877 NaN -0.061234 NaN   
                    EqualWeight_Rebalanced -0.055871 NaN -0.058111 NaN   
A4_QA_only          BuyHold_EqualInitial   -0.076315 NaN -0.079340 NaN   
                    EqualWeight_Rebalanced -0.055718 NaN -0.057953 NaN   
A5_KD_only          BuyHold_EqualInitial   -0.058667 NaN -0.061016 NaN   
                    EqualWeight_Rebalanced -0.058042 NaN -0.060367 NaN   
A6_SH_QA            BuyHold_EqualInitial   -0.025656 NaN -0.026702 NaN   
                    EqualWeight_Rebalanced -0.007408 NaN -0.007713 NaN   
A7_SH_KD            BuyHold_EqualInitial   -0.041538 NaN -0.043217 NaN   
                    EqualWeight_Rebalanced -0.041240 NaN -0.042907 NaN   
A8_QA_KD            BuyHold_EqualInitial   -0.099672 NaN -0.103570 NaN   
                    EqualWeight_Rebalanced -0.075446 NaN -0.078439 NaN   
A9_Fixed_fusion     BuyHold_EqualInitial   -0.052966 NaN -0.055093 NaN   
                    EqualWeight_Rebalanced -0.007331 NaN -0.007633 NaN   

                                                 Vol            SR      \
                                                mean std      mean std   
variant             agent                                                
A0_Full_MGT         BuyHold_EqualInitial    0.134256 NaN  0.251783 NaN   
                    EqualWeight_Rebalanced  0.134722 NaN  0.521504 NaN   
A1_Transformer_only BuyHold_EqualInitial    0.031825 NaN -0.620783 NaN   
                    EqualWeight_Rebalanced  0.032308 NaN -0.592338 NaN   
A2_MGNN_only        BuyHold_EqualInitial    0.204793 NaN -0.063770 NaN   
                    EqualWeight_Rebalanced  0.201579 NaN  0.406283 NaN   
A3_SH_only          BuyHold_EqualInitial    0.087679 NaN -0.698386 NaN   
                    EqualWeight_Rebalanced  0.089112 NaN -0.652117 NaN   
A4_QA_only          BuyHold_EqualInitial    0.111273 NaN -0.713022 NaN   
                    EqualWeight_Rebalanced  0.115399 NaN -0.502193 NaN   
A5_KD_only          BuyHold_EqualInitial    0.052742 NaN -1.156870 NaN   
                    EqualWeight_Rebalanced  0.054897 NaN -1.099647 NaN   
A6_SH_QA            BuyHold_EqualInitial    0.138037 NaN -0.193444 NaN   
                    EqualWeight_Rebalanced  0.139357 NaN -0.055345 NaN   
A7_SH_KD            BuyHold_EqualInitial    0.041928 NaN -1.030747 NaN   
                    EqualWeight_Rebalanced  0.043301 NaN -0.990912 NaN   
A8_QA_KD            BuyHold_EqualInitial    0.138099 NaN -0.749967 NaN   
                    EqualWeight_Rebalanced  0.148407 NaN -0.528536 NaN   
A9_Fixed_fusion     BuyHold_EqualInitial    0.177549 NaN -0.310299 NaN   
                    EqualWeight_Rebalanced  0.198771 NaN -0.038401 NaN   

                                                 MDD        Calmar      \
                                                mean std      mean std   
variant             agent                                                
A0_Full_MGT         BuyHold_EqualInitial   -0.094951 NaN  0.356008 NaN   
                    EqualWeight_Rebalanced -0.081513 NaN  0.861924 NaN   
A1_Transformer_only BuyHold_EqualInitial   -0.050688 NaN -0.389773 NaN   
                    EqualWeight_Rebalanced -0.051036 NaN -0.374974 NaN   
A2_MGNN_only        BuyHold_EqualInitial   -0.111082 NaN -0.117567 NaN   
                    EqualWeight_Rebalanced -0.081706

In [14]:
# ===============================
# 9.1 Paper-ready tables for A0-A9 × five RL agents and best-policy table
# ===============================

def _format_method_name(x: str) -> str:
    return str(x).replace("_", " ").replace("SH QA", "SH+QA").replace("SH KD", "SH+KD").replace("QA KD", "QA+KD")

def _format_policy_name(x: str) -> str:
    mapping = {
        "a2c": "A2C", "ppo": "PPO", "sac": "SAC", "td3": "TD3", "ddpg": "DDPG",
        "BuyHold_EqualInitial": "Buy-and-Hold",
        "EqualWeight_Rebalanced": "Equal Weight",
    }
    return mapping.get(str(x), str(x))

def write_best_policy_latex(best_df: pd.DataFrame, out_path: Path, caption: str, label: str):
    df = best_df.copy()
    df["Method"] = df["variant"].map(_format_method_name)
    df["Best policy"] = df["agent"].map(_format_policy_name)
    # best columns: larger is better except MDD closer to zero is better
    best_max_cols = ["CR", "AR", "SR", "Calmar", "Sortino", "FinalAccountValue"]
    best_min_abs_col = "MDD"
    best_idx = {c: df[c].idxmax() for c in best_max_cols if c in df.columns}
    if best_min_abs_col in df.columns:
        best_idx[best_min_abs_col] = df[best_min_abs_col].abs().idxmin()

    def fmt(col, val, idx):
        if col == "FinalAccountValue":
            text = f"{val:,.0f}"
        else:
            text = f"{val:.4f}"
        if best_idx.get(col, None) == idx:
            return r"\textbf{" + text + "}"
        return text

    lines = []
    lines.append(r"\begin{table*}[t]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\resizebox{\textwidth}{!}{")
    lines.append(r"\begin{tabular}{llccccccc}")
    lines.append(r"\toprule")
    lines.append(r"Method & Best policy & CR & AR & SR & MDD & Calmar & Sortino & Final value \\")
    lines.append(r"\midrule")
    for idx, row in df.iterrows():
        lines.append(
            f"{row['Method']} & {row['Best policy']} & "
            f"{fmt('CR', row['CR'], idx)} & {fmt('AR', row['AR'], idx)} & {fmt('SR', row['SR'], idx)} & "
            f"{fmt('MDD', row['MDD'], idx)} & {fmt('Calmar', row['Calmar'], idx)} & "
            f"{fmt('Sortino', row['Sortino'], idx)} & {fmt('FinalAccountValue', row['FinalAccountValue'], idx)} \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}")
    lines.append(r"\vspace{1mm}")
    lines.append(r"\parbox{\textwidth}{\footnotesize Note: For each ablation variant, the reported row corresponds to the best downstream allocation policy according to final account value. CR, AR, SR, MDD, Calmar, and Sortino denote cumulative return, annualized return, Sharpe ratio, maximum drawdown, Calmar ratio, and Sortino ratio, respectively. For MDD, a value closer to zero indicates a smaller drawdown.}")
    lines.append(r"\end{table*}")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    print("Saved LaTeX table:", out_path)

if len(raw_metrics_df) == 0:
    print("No results. Skip paper-ready tables.")
else:
    paper_cols = [
        "variant", "agent", "seed", "CR", "AR", "Vol", "SR", "MDD",
        "Calmar", "Sortino", "Omega", "Turnover", "FinalAccountValue"
    ]
    exist_cols = [c for c in paper_cols if c in raw_metrics_df.columns]
    paper_table = raw_metrics_df[exist_cols].copy()

    metric_cols = ["CR", "AR", "Vol", "SR", "MDD", "Calmar", "Sortino", "Omega", "Turnover", "FinalAccountValue"]
    metric_cols = [c for c in metric_cols if c in paper_table.columns]

    paper_mean = (
        paper_table
        .groupby(["variant", "agent"], as_index=False)[metric_cols]
        .mean()
        .sort_values(["variant", "agent"])
    )

    paper_table_path = RL_ABLATION_DIR / "rl_ablation_paper_table.csv"
    paper_mean.to_csv(paper_table_path, index=False, encoding="utf-8-sig")
    print("Saved paper table:", paper_table_path)

    pivot_final = paper_mean.pivot_table(
        index="variant",
        columns="agent",
        values="FinalAccountValue",
        aggfunc="mean"
    )
    pivot_final_path = RL_ABLATION_DIR / "pivot_final_account_value_A0_A9_agents.csv"
    pivot_final.to_csv(pivot_final_path, encoding="utf-8-sig")
    print("Saved final value pivot:", pivot_final_path)

    pivot_sharpe = paper_mean.pivot_table(
        index="variant",
        columns="agent",
        values="SR",
        aggfunc="mean"
    )
    pivot_sharpe_path = RL_ABLATION_DIR / "pivot_sharpe_A0_A9_agents.csv"
    pivot_sharpe.to_csv(pivot_sharpe_path, encoding="utf-8-sig")
    print("Saved Sharpe pivot:", pivot_sharpe_path)

    best_agent_by_variant = (
        paper_mean.sort_values("FinalAccountValue", ascending=False)
        .groupby("variant", as_index=False)
        .head(1)
        .sort_values("variant")
    )
    best_path = RL_ABLATION_DIR / "best_agent_by_variant.csv"
    best_agent_by_variant.to_csv(best_path, index=False, encoding="utf-8-sig")
    print("Saved best agent table:", best_path)

    best_latex_path = RL_ABLATION_DIR / "bank_forecast_to_policy_best_policy_table.tex"
    write_best_policy_latex(
        best_agent_by_variant,
        best_latex_path,
        caption="Forecast-to-policy ablation results on the banking sector.",
        label="tab:bank_forecast_policy_ablation",
    )

    display(paper_mean)
    display(pivot_final)
    display(pivot_sharpe)
    display(best_agent_by_variant)


Saved paper table: D:\paper\onging\ablation\rl_bank_best_policy\rl_ablation_paper_table.csv
Saved final value pivot: D:\paper\onging\ablation\rl_bank_best_policy\pivot_final_account_value_A0_A9_agents.csv
Saved Sharpe pivot: D:\paper\onging\ablation\rl_bank_best_policy\pivot_sharpe_A0_A9_agents.csv
Saved best agent table: D:\paper\onging\ablation\rl_bank_best_policy\best_agent_by_variant.csv
Saved LaTeX table: D:\paper\onging\ablation\rl_bank_best_policy\bank_forecast_to_policy_best_policy_table.tex


,variant,agent,CR,AR,Vol,SR,MDD,Calmar,Sortino,Omega,Turnover,FinalAccountValue
0,A0_Full_MGT,BuyHold_EqualInitial,0.032440,0.033803,0.134256,0.251783,-0.094951,0.356008,0.391118,1.061263,NaN,1.032440e+06
1,A0_Full_MGT,EqualWeight_Rebalanced,0.067378,0.070258,0.134722,0.521504,-0.081513,0.861924,0.834250,1.114313,NaN,1.067378e+06
2,A1_Transformer_only,BuyHold_EqualInitial,-0.018980,-0.019757,0.031825,-0.620783,-0.050688,-0.389773,-0.977704,0.903164,NaN,9.810199e+05
3,A1_Transformer_only,EqualWeight_Rebalanced,-0.018385,-0.019137,0.032308,-0.592338,-0.051036,-0.374974,-0.936791,0.907433,NaN,9.816151e+05
4,A2_MGNN_only,BuyHold_EqualInitial,-0.012545,-0.013060,0.204793,-0.063770,-0.111082,-0.117567,-0.088290,1.006827,NaN,9.874554e+05
5,A2_MGNN_only,EqualWeight_Rebalanced,0.078524,0.081898,0.201579,0.406283,-0.081706,1.002354,0.572866,1.092955,NaN,1.078524e+06
6,A3_SH_only,BuyHold_EqualInitial,-0.058877,-0.061234,0.087679,-0.698386,-0.140163,-0.436877,-1.088600,0.894062,NaN,9.411229e+05
7,A3_SH_only,EqualWeight_Rebalanced,-0.055871,-0.058111,0.089112,-0.652117,-0.140674,-0.413093,-1.022832,0.901343,NaN,9.441289e+05
8,A4_QA_only,BuyHold_EqualInitial,-0.076315,-0.079340,0.111273,-0.713022,-0.150357,-0.527679,-1.118736,0.889598,NaN,9.236850e+05
9,A4_QA_only,EqualWeight_Rebalanced,-0.055718,-0.057953,0.115399,-0.502193,-0.140914,-0.411262,-0.797862,0.924819,NaN,9.442816e+05


agent,BuyHold_EqualInitial,EqualWeight_Rebalanced
variant,,
A0_Full_MGT,1.032440e+06,1.067378e+06
A1_Transformer_only,9.810199e+05,9.816151e+05
A2_MGNN_only,9.874554e+05,1.078524e+06
A3_SH_only,9.411229e+05,9.441289e+05
A4_QA_only,9.236850e+05,9.442816e+05
A5_KD_only,9.413332e+05,9.419577e+05
A6_SH_QA,9.743435e+05,9.925922e+05
A7_SH_KD,9.584620e+05,9.587599e+05
A8_QA_KD,9.003279e+05,9.245535e+05


agent,BuyHold_EqualInitial,EqualWeight_Rebalanced
variant,,
A0_Full_MGT,0.251783,0.521504
A1_Transformer_only,-0.620783,-0.592338
A2_MGNN_only,-0.063770,0.406283
A3_SH_only,-0.698386,-0.652117
A4_QA_only,-0.713022,-0.502193
A5_KD_only,-1.156870,-1.099647
A6_SH_QA,-0.193444,-0.055345
A7_SH_KD,-1.030747,-0.990912
A8_QA_KD,-0.749967,-0.528536


,variant,agent,CR,AR,Vol,SR,MDD,Calmar,Sortino,Omega,Turnover,FinalAccountValue
1,A0_Full_MGT,EqualWeight_Rebalanced,0.067378,0.070258,0.134722,0.521504,-0.081513,0.861924,0.834250,1.114313,NaN,1.067378e+06
3,A1_Transformer_only,EqualWeight_Rebalanced,-0.018385,-0.019137,0.032308,-0.592338,-0.051036,-0.374974,-0.936791,0.907433,NaN,9.816151e+05
5,A2_MGNN_only,EqualWeight_Rebalanced,0.078524,0.081898,0.201579,0.406283,-0.081706,1.002354,0.572866,1.092955,NaN,1.078524e+06
7,A3_SH_only,EqualWeight_Rebalanced,-0.055871,-0.058111,0.089112,-0.652117,-0.140674,-0.413093,-1.022832,0.901343,NaN,9.441289e+05
9,A4_QA_only,EqualWeight_Rebalanced,-0.055718,-0.057953,0.115399,-0.502193,-0.140914,-0.411262,-0.797862,0.924819,NaN,9.442816e+05
11,A5_KD_only,EqualWeight_Rebalanced,-0.058042,-0.060367,0.054897,-1.099647,-0.097382,-0.619900,-1.543529,0.824704,NaN,9.419577e+05
13,A6_SH_QA,EqualWeight_Rebalanced,-0.007408,-0.007713,0.139357,-0.055345,-0.096922,-0.079577,-0.072943,1.002643,NaN,9.925922e+05
15,A7_SH_KD,EqualWeight_Rebalanced,-0.041240,-0.042907,0.043301,-0.990912,-0.074147,-0.578678,-1.505132,0.847008,NaN,9.587599e+05
17,A8_QA_KD,EqualWeight_Rebalanced,-0.075446,-0.078439,0.148407,-0.528536,-0.194040,-0.404239,-0.781779,0.920054,NaN,9.245535e+05
19,A9_Fixed_fusion,EqualWeight_Rebalanced,-0.007331,-0.007633,0.198771,-0.038401,-0.128719,-0.059299,-0.048755,1.013478,NaN,9.926688e+05


## 10. 画累计收益曲线

生成用于论文的 SVG 图：

```text
D:/paper/onging/ablation/rl/rl_ablation_cumulative_return.svg
```

建议放在 `Ablation study on forecast-driven portfolio allocation` 小节。

In [15]:
# ===============================
# 10. Plot cumulative return curves
# ===============================

def plot_cumulative_return_curves(results_dir=RL_ABLATION_DIR):
    results_dir = Path(results_dir)
    files = sorted(results_dir.glob("*_daily_return.csv"))

    if not files:
        print("No daily_return files found.")
        return

    plt.figure(figsize=(9, 5))

    for file in files:
        df_ret = pd.read_csv(file)
        if "daily_return" not in df_ret.columns:
            continue

        label = file.stem.replace("_daily_return", "")
        curve = (1.0 + df_ret["daily_return"].astype(float)).cumprod() - 1.0
        plt.plot(curve.values, label=label, linewidth=1.5)

    plt.xlabel("Trading day")
    plt.ylabel("Cumulative return")
    plt.legend(fontsize=7)
    plt.tight_layout()

    out_path = results_dir / "rl_ablation_cumulative_return.svg"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)

plot_cumulative_return_curves()


Saved: D:\paper\onging\ablation\rl_bank_best_policy\rl_ablation_cumulative_return.svg


C:\Users\perfe\AppData\Local\Temp\ipykernel_16268\2609729345.py:31: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 11. 可选：完整 RL agent 比较

如果你要做完整策略比较，可以把下面的 `RUN_FULL_AGENT_COMPARISON` 改成 `True`。

这部分只建议对 `A0_Full_MGT` 运行 PPO/A2C/SAC/TD3/DDPG，不建议所有消融版本都跑五个 agent，计算量太大。

In [16]:
# ===============================
# 11. Optional full-agent comparison
# ===============================

RUN_FULL_AGENT_COMPARISON = False

if RUN_FULL_AGENT_COMPARISON:
    full_agent_metrics = []
    variant_tag = "A0_Full_MGT"
    csv_path = RL_ABLATION_INPUTS[variant_tag]

    FULL_AGENTS = ["a2c", "ppo", "sac", "td3", "ddpg"]

    for agent_name in FULL_AGENTS:
        for seed in SEEDS:
            metrics = train_and_trade_one_agent(
                csv_path=csv_path,
                variant_tag=variant_tag,
                agent_name=agent_name,
                seed=seed,
            )
            full_agent_metrics.append(metrics)

    full_agent_df = pd.DataFrame(full_agent_metrics)
    full_agent_path = RL_ABLATION_DIR / "full_agent_comparison_raw_results.csv"
    full_agent_df.to_csv(full_agent_path, index=False)

    print("Saved:", full_agent_path)
    display(full_agent_df)
else:
    print("RUN_FULL_AGENT_COMPARISON = False; skip full PPO/A2C/SAC/TD3/DDPG comparison.")


RUN_FULL_AGENT_COMPARISON = False; skip full PPO/A2C/SAC/TD3/DDPG comparison.


## 12. 可选：从第一个预测消融结果生成 FinRL 长表 CSV 的辅助函数

如果第一个 notebook 输出的是 `A*_pred.csv` 预测矩阵，而不是 FinRL 长表 CSV，可以用下面函数把预测 close 合并回原始 OHLCV 长表中。

注意：

1. 预测矩阵必须已经反归一化到真实价格尺度；
2. 预测矩阵的列顺序必须与 `base_trade_csv` 中的 ticker 顺序一致，或者直接带有 ticker 列名；
3. 预测行数必须与指定日期范围内的交易日数量一致。

如果你的预测 CSV 仍然是 MinMaxScaler 后的 `[-1, 1]` 数值，必须先反归一化，否则不能用于回测。

In [18]:
# ===============================
# 12. Optional helper: merge predicted close into FinRL long table
# ===============================

def merge_predicted_close_to_finrl_csv(
    base_trade_csv,
    pred_matrix_csv,
    output_csv,
    start_date=TRADE_START,
    end_date=TRADE_END,
    tickers=None,
    pred_is_real_price=True,
):
    if not pred_is_real_price:
        raise ValueError("预测矩阵仍是归一化值，必须先反归一化到真实价格尺度。")

    base_trade_csv = Path(base_trade_csv)
    pred_matrix_csv = Path(pred_matrix_csv)
    output_csv = Path(output_csv)

    base = pd.read_csv(base_trade_csv)
    _check_required_columns(base, base_trade_csv)

    base["date"] = pd.to_datetime(base["date"]).dt.strftime("%Y-%m-%d")
    date_mask = (base["date"] >= start_date) & (base["date"] <= end_date)

    if tickers is None:
        tickers = sorted(base["tic"].unique().tolist())

    dates = sorted(base.loc[date_mask, "date"].unique().tolist())

    pred = pd.read_csv(pred_matrix_csv)

    # 如果预测 CSV 多了一列 unnamed index，则删除
    unnamed = [c for c in pred.columns if str(c).lower().startswith("unnamed")]
    if unnamed:
        pred = pred.drop(columns=unnamed)

    if pred.shape[1] == len(tickers):
        pred.columns = tickers
    else:
        missing = set(tickers) - set(pred.columns)
        if missing:
            raise ValueError(
                f"预测矩阵列数/列名与 ticker 不匹配。pred shape={pred.shape}, "
                f"num tickers={len(tickers)}, missing={missing}"
            )
        pred = pred[tickers]

    if len(pred) != len(dates):
        raise ValueError(
            f"预测矩阵行数 {len(pred)} 与日期数量 {len(dates)} 不一致。"
            f"请检查 start_date/end_date 或预测输出区间。"
        )

    pred = pred.copy()
    pred["date"] = dates
    pred_long = pred.melt(id_vars="date", var_name="tic", value_name="pred_close")

    merged = base.merge(pred_long, on=["date", "tic"], how="left")
    replace_mask = merged["pred_close"].notna()

    merged.loc[replace_mask, "close"] = merged.loc[replace_mask, "pred_close"]
    merged = merged.drop(columns=["pred_close"])

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output_csv, index=False)

    print("Saved merged FinRL CSV:", output_csv)
    print("Replaced close rows:", int(replace_mask.sum()))

    return merged

# 示例，不自动运行：
# merge_predicted_close_to_finrl_csv(
#     base_trade_csv=BASE_FULL_MGT_CSV,
#     pred_matrix_csv=ABLATION_ROOT / "A1_Transformer_only_seed1_pred_real_price.csv",
#     output_csv=GRAPH_DATA_DIR / "banktrademergeGNNtr20241231_A1_Transformer_only.CSV",
#     start_date=TRADE_START,
#     end_date=TRADE_END,
# )


## 13. 论文中放置位置

本 notebook 产生的结果建议放在：

```latex
\subsection{Ablation study on forecast-driven portfolio allocation}
```

表格建议命名：

```latex
Table X. Forecast-to-policy ablation under the same RL agent.
```

推荐表头：

```latex
State signal & RL agent & AR & SR & MDD & Calmar & Turnover
```

正文解释重点：

1. 固定 RL agent，只改变 forecast-driven state；
2. Full MGT-FinRL 优于 Vanilla/ablation signals，说明收益来自 MGT 的关系-时间预测信号；
3. Transformer-only 或 MGNN-only 下降，说明单独时间建模或单独图建模都不充分；
4. 单图版本弱于 Full MGT，说明 SH/QA/KD 三类关系存在互补性。

## 输出文件说明

运行完成后，重点查看以下文件：

```text
D:/paper/onging/ablation/rl/rl_ablation_paper_table.csv
```

这是论文中完整策略层消融表：`A0-A9 × A2C/PPO/SAC/TD3/DDPG`。

如果表太大，正文建议只放：

```text
D:/paper/onging/ablation/rl/best_agent_by_variant.csv
```

完整表可以放 Appendix。

如果自动生成 FinRL CSV 时提示“预测矩阵行数与指定日期区间不一致”，说明第一个预测消融实验输出的 `pred.csv` 日期范围和本 notebook 中的 `PREDICTION_START/PREDICTION_END` 不一致。请先确认第一个 notebook 的测试区间，然后修改第 2 节的日期。
